# Notebook 4 — Automation v2 (Real-Time Pipeline)

**What's new in v2:**
- Verified source list (CNN / WSJ / Forbes removed)
- 48-hour date filter on every cycle
- Similarity deduplication across the full live dataset (not just new articles)
- Section assigned to every new article
- Per-section live feed viewer at the end

**Output:** `smart_news_live.csv` — continuously updated

In [ ]:
# %pip install apify-client transformers torch pandas schedule sentencepiece scikit-learn python-dateutil

In [ ]:
from apify_client import ApifyClient
import pandas as pd
import torch
import schedule
import time
import os
from datetime import datetime, timezone
from dateutil import parser as dateparser
from transformers import pipeline as hf_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print('Libraries loaded.')

In [ ]:
# ═══════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════
APIFY_API_TOKEN         = 'YOUR_APIFY_API_TOKEN_HERE'
ACTOR_ID                = 'complex_intricate_networks/news-article-scraper-100-global-sources-api'
OUTPUT_CSV              = 'smart_news_live.csv'
FETCH_INTERVAL_MINUTES  = 30
MAX_ARTICLES_PER_SOURCE = 100
MAX_AGE_HOURS           = 48
SIMILARITY_THRESHOLD    = 0.70
MODEL_NAME              = 'sshleifer/distilbart-cnn-12-6'

# Verified working sources only
SOURCES = [
    'BBC World News', 'Al Jazeera', 'CNBC', 'The New York Times',
    'TechCrunch', 'Wired', 'Variety', 'The Guardian World News',
    'Business Insider', 'Mashable', 'Rolling Stone', 'Vox',
    'TIME', 'Ars Technica', 'CNET News', 'CBS News', 'CoinDesk',
]

SECTION_KEYWORDS = {
    'Tech': ['ai', 'artificial intelligence', 'machine learning', 'software', 'app',
             'startup', 'google', 'apple', 'microsoft', 'meta', 'openai', 'nvidia',
             'chip', 'cybersecurity', 'hack', 'data breach', 'electric vehicle', 'ev',
             'tesla', 'smartphone', 'iphone', 'android', 'tech', 'technology',
             'developer', 'chatgpt', 'llm', 'drone', '5g', 'quantum', 'computer'],
    'Politics': ['election', 'president', 'prime minister', 'parliament', 'congress',
                 'senate', 'government', 'policy', 'democrat', 'republican', 'labour',
                 'trump', 'biden', 'vote', 'ballot', 'legislation', 'minister',
                 'cabinet', 'diplomat', 'sanctions', 'nato', 'united nations',
                 'political', 'politician', 'campaign', 'opposition', 'albanese'],
    'Business': ['stock', 'market', 'shares', 'economy', 'inflation', 'interest rate',
                 'gdp', 'recession', 'investment', 'revenue', 'profit', 'earnings',
                 'merger', 'bankruptcy', 'layoff', 'unemployment', 'trade', 'tariff',
                 'finance', 'bank', 'crypto', 'bitcoin', 'wall street', 'asx', 'nasdaq'],
    'Crime': ['murder', 'killed', 'arrested', 'charged', 'sentenced', 'prison',
              'jail', 'court', 'trial', 'verdict', 'police', 'shooting', 'stabbing',
              'robbery', 'fraud', 'scam', 'trafficking', 'terrorist', 'bomb',
              'hostage', 'kidnap', 'homicide', 'assault', 'corruption'],
    'Science': ['research', 'study', 'scientists', 'discovery', 'space', 'nasa',
                'climate change', 'environment', 'species', 'gene', 'dna', 'vaccine',
                'virus', 'bacteria', 'cancer', 'pandemic', 'outbreak', 'disease',
                'astronomy', 'planet', 'telescope', 'reef', 'earthquake', 'volcano'],
    'Sport': ['match', 'tournament', 'championship', 'league', 'cup', 'final',
              'score', 'goal', 'player', 'coach', 'transfer', 'football', 'soccer',
              'rugby', 'cricket', 'tennis', 'golf', 'basketball', 'nba', 'nfl',
              'nrl', 'afl', 'f1', 'formula 1', 'olympics', 'athlete', 'motogp'],
    'Entertainment': ['movie', 'film', 'box office', 'oscar', 'bafta', 'emmy', 'grammy',
                      'celebrity', 'actor', 'actress', 'album', 'concert', 'streaming',
                      'netflix', 'disney', 'hbo', 'tv show', 'series', 'award',
                      'singer', 'band', 'music', 'pop', 'hip hop', 'viral', 'tiktok'],
    'Lifestyle': ['food', 'recipe', 'restaurant', 'travel', 'holiday', 'vacation',
                  'wellness', 'mental health', 'fitness', 'workout', 'diet', 'parenting',
                  'relationship', 'dating', 'wedding', 'home', 'fashion', 'beauty',
                  'skincare', 'shopping', 'review', 'how to', 'tips', 'advice'],
    'World': ['war', 'conflict', 'military', 'troops', 'attack', 'strike', 'missile',
              'ceasefire', 'humanitarian', 'refugee', 'ukraine', 'russia', 'israel',
              'gaza', 'iran', 'china', 'north korea', 'middle east', 'africa',
              'europe', 'asia', 'flood', 'disaster', 'protest', 'coup'],
}
SECTION_PRIORITY = ['Crime', 'Tech', 'Politics', 'Business', 'Science',
                    'Sport', 'Entertainment', 'Lifestyle', 'World']

PROMO_KEYWORDS   = ['promo code', 'coupon', '% off', 'discount', 'gift guide']
BAD_URL_KEYWORDS = ['/video/', '/live-news/', '/category/', '/index',
                    '/watch', '/gallery/', '/podcast/', '/audio/', '/live/']

VALID_ACTOR_SOURCES = [
    'The Times of India','GeeksforGeeks','Fandom','Hindustan Times','Economic Times',
    'Moneycontrol','The Indian Express','Mint','The Hindu','Investopedia','India Today',
    'ESPN Cricinfo','NDTV','Healthline','Forbes','MakeUseOf','TechTarget','BBC World News',
    'Business Standard','The Guardian World News','News18','Gadgets 360','Business Today',
    'ABP News','GSMArena','The New York Times','HowToGeek','XDA Developers','TechRadar',
    'IGN Reviews','Nature','SamMobile','India.com','Lifewire','Screenrant','Bloomberg',
    'Gadgets Now','PCMag UK','Tech Advisor','CNET News','Business Insider','Game Rant',
    'Times Now','Techspot','India TV','Filmibeat','Al Jazeera','Collider','CNBC',
    'Techopedia','CBR',"Tom's Guide",'Windows Report','CNN','Amar Ujala','Prokerala',
    'Android Authority','Oneindia','Beebom','Deccan Herald','Anime News Network India',
    'WSJ World News','Zee Business','Variety','Goodreturns','CNBCTV18','Windows Central',
    'Very Well Mind','Harvard Business Review','The New Indian Express','Very Well Health',
    'Wired',"Tom's Hardware",'Washington Post World News','Analytics India Magazine',
    'Cosmopolitan','TheGamer','Myupchar','YourStory','Firstpost','Games Radar','Scroll.in',
    'The Windows Club','ThePrint','Financial Times','TIME','TechCrunch','Vogue India',
    'PC Gamer','DNA India','The Quint','Deccan Chronicle','Polygon','The Hollywood Reporter',
    "It's FOSS",'Pocket-lint','Pinkvilla','Mashable','CBS News','Bollywood Hungama','Digit',
    'Webdunia English','Dot Esports','Help Desk Geek','Eurogamer','The New Yorker','TecMint',
    'Rolling Stone','Ars Technica','CoinDesk','Search Engine Land','GQ India','Onmanorama',
    'Republic World','Fextralife','Free Press Journal','Daily Mail India','Vox',
    'Rock Paper Shotgun','ArchDaily','Deadline','Billboard','The Economist','Make Tech Easier',
    'The Week','The Atlantic','The News Minute','Welcome to Good Food',
    'Sportstar','Independent Asia','ANI','Brave Blog','ESPN India'
]

In [ ]:
# Load model once — reused across all cycles
device     = 0 if torch.cuda.is_available() else -1
print(f'Loading {MODEL_NAME} on {"GPU" if device == 0 else "CPU"}...')
summariser = hf_pipeline('summarization', model=MODEL_NAME, device=device)
apify      = ApifyClient(APIFY_API_TOKEN)
print('Ready.')

In [ ]:
# ═══════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════

def classify_section(title, description):
    text = (title + ' ' + description).lower()
    for section in SECTION_PRIORITY:
        if any(kw in text for kw in SECTION_KEYWORDS[section]):
            return section
    return 'World'

def parse_dt(raw):
    if not raw or not isinstance(raw, str): return None
    try:
        dt = dateparser.parse(raw)
        return dt.replace(tzinfo=timezone.utc) if dt and not dt.tzinfo else dt
    except: return None

def age_hours(dt):
    if dt is None: return 9999
    return (datetime.now(timezone.utc) - dt).total_seconds() / 3600

def is_valid(item):
    url   = (item.get('url') or '').lower()
    title = (item.get('title') or '').lower().strip()
    desc  = (item.get('description') or '')
    if any(kw in url for kw in BAD_URL_KEYWORDS): return False
    if len(desc) < 20:                             return False
    if any(kw in title for kw in PROMO_KEYWORDS): return False
    return True

def summarise(text):
    if not isinstance(text, str) or len(text.strip()) < 50:
        return 'Summary not available.'
    try:
        out   = summariser(text[:3000], max_length=130, min_length=30, do_sample=False)
        words = out[0]['summary_text'].strip().split()
        return ' '.join(words[:60]) + ('...' if len(words) > 60 else '')
    except Exception as e:
        return f'Error: {str(e)[:50]}'

def load_seen_urls():
    if os.path.exists(OUTPUT_CSV):
        return set(pd.read_csv(OUTPUT_CSV, usecols=['url'])['url'].dropna())
    return set()

def append_to_csv(rows):
    new_df = pd.DataFrame(rows)
    if os.path.exists(OUTPUT_CSV):
        new_df.to_csv(OUTPUT_CSV, mode='a', header=False, index=False, encoding='utf-8')
    else:
        new_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
    print(f'  Appended {len(rows)} new articles to {OUTPUT_CSV}')

def similarity_dedup_against_existing(new_rows, existing_urls):
    """
    Among new_rows only, remove near-duplicates of each other.
    (Existing articles already in CSV are handled by URL dedup.)
    """
    if len(new_rows) < 2:
        return new_rows
    df_new  = pd.DataFrame(new_rows)
    texts   = (df_new['title'].fillna('') + ' ' + df_new['description'].fillna('')).tolist()
    vec     = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 2))
    tfidf   = vec.fit_transform(texts)
    sim     = cosine_similarity(tfidf)
    to_drop = set()
    for i in range(len(df_new)):
        if i in to_drop: continue
        for j in range(i + 1, len(df_new)):
            if j not in to_drop and sim[i, j] >= SIMILARITY_THRESHOLD:
                to_drop.add(j)
    kept = [r for idx, r in enumerate(new_rows) if idx not in to_drop]
    if to_drop:
        print(f'  Similarity dedup: {len(new_rows)} -> {len(kept)} ({len(to_drop)} near-duplicates removed)')
    return kept

In [ ]:
# ═══════════════════════════════════════════════════════
# MAIN CYCLE
# ═══════════════════════════════════════════════════════

def run_cycle():
    print(f'\n{"="*60}')
    print(f'Cycle: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
    print(f'{"="*60}')

    seen_urls       = load_seen_urls()
    filtered_srcs   = [s for s in SOURCES if s in VALID_ACTOR_SOURCES]

    # ── Fetch from Apify ──────────────────────────────────
    try:
        run   = apify.actor(ACTOR_ID).call(run_input={
            'category': 'Top news', 'countries': ['Australia'],
            'sources': filtered_srcs, 'maxArticles': MAX_ARTICLES_PER_SOURCE,
            'keywordFilter': '', 'proxyConfiguration': {'useApifyProxy': True}
        })
        items = list(apify.dataset(run['defaultDatasetId']).iterate_items())
        print(f'Apify returned {len(items)} raw articles.')
    except Exception as e:
        print(f'Apify error: {e}'); return

    # ── Filter + classify ─────────────────────────────────
    new_rows = []
    for item in items:
        url = item.get('url', '')
        if not url or url in seen_urls: continue
        if not is_valid(item):          continue

        pub_dt  = parse_dt(item.get('published_at', ''))
        hrs_old = age_hours(pub_dt)
        if hrs_old > MAX_AGE_HOURS:     continue   # 48-hour filter

        title = (item.get('title') or '').strip()
        desc  = (item.get('description') or '').strip()

        new_rows.append({
            'website_name': item.get('source', '').strip(),
            'section':      classify_section(title, desc),
            'title':        title,
            'description':  desc,
            'ai_summary':   '',
            'url':          url,
            'published_at': pub_dt.strftime('%Y-%m-%d %H:%M UTC') if pub_dt else '',
            'age_hours':    round(hrs_old, 1),
            'scrape_time':  datetime.now().strftime('%H:%M:%S'),
        })
        seen_urls.add(url)

    print(f'New articles (after URL + date filter): {len(new_rows)}')

    # ── Similarity dedup among new articles ──────────────
    new_rows = similarity_dedup_against_existing(new_rows, seen_urls)

    if not new_rows:
        print('No new articles this cycle.')
        print(f'Next cycle in {FETCH_INTERVAL_MINUTES} min.')
        return

    # ── Summarise ─────────────────────────────────────────
    for i, row in enumerate(new_rows):
        row['ai_summary'] = summarise(row['description'])
        print(f"  [{i+1}/{len(new_rows)}] [{row['section']}] {row['title'][:55]}")
        print(f"        {row['ai_summary'][:80]}...")

    # ── Save to CSV ───────────────────────────────────────
    append_to_csv(new_rows)

    # ── Push to Supabase (if enabled) ─────────────────────
    write_to_supabase(new_rows)

    print(f'Total in dataset: {len(load_seen_urls())}')
    print(f'Next cycle in {FETCH_INTERVAL_MINUTES} min.')

In [ ]:
# ═══════════════════════════════════════════════════════
# MAIN CYCLE
# ═══════════════════════════════════════════════════════

def run_cycle():
    print(f'\n{"="*60}')
    print(f'Cycle: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
    print(f'{"="*60}')

    seen_urls       = load_seen_urls()
    filtered_srcs   = [s for s in SOURCES if s in VALID_ACTOR_SOURCES]

    # ── Fetch from Apify ──────────────────────────────────
    try:
        run   = apify.actor(ACTOR_ID).call(run_input={
            'category': 'Top news', 'countries': ['Australia'],
            'sources': filtered_srcs, 'maxArticles': MAX_ARTICLES_PER_SOURCE,
            'keywordFilter': '', 'proxyConfiguration': {'useApifyProxy': True}
        })
        items = list(apify.dataset(run['defaultDatasetId']).iterate_items())
        print(f'Apify returned {len(items)} raw articles.')
    except Exception as e:
        print(f'Apify error: {e}'); return

    # ── Filter + classify ─────────────────────────────────
    new_rows = []
    for item in items:
        url = item.get('url', '')
        if not url or url in seen_urls: continue
        if not is_valid(item):          continue

        pub_dt  = parse_dt(item.get('published_at', ''))
        hrs_old = age_hours(pub_dt)
        if hrs_old > MAX_AGE_HOURS:     continue   # 48-hour filter

        title = (item.get('title') or '').strip()
        desc  = (item.get('description') or '').strip()

        new_rows.append({
            'website_name': item.get('source', '').strip(),
            'section':      classify_section(title, desc),
            'title':        title,
            'description':  desc,
            'ai_summary':   '',
            'url':          url,
            'published_at': pub_dt.strftime('%Y-%m-%d %H:%M UTC') if pub_dt else '',
            'age_hours':    round(hrs_old, 1),
            'scrape_time':  datetime.now().strftime('%H:%M:%S'),
        })
        seen_urls.add(url)

    print(f'New articles (after URL + date filter): {len(new_rows)}')

    # ── Similarity dedup among new articles ──────────────
    new_rows = similarity_dedup_against_existing(new_rows, seen_urls)

    if not new_rows:
        print('No new articles this cycle.')
        print(f'Next cycle in {FETCH_INTERVAL_MINUTES} min.')
        return

    # ── Summarise ─────────────────────────────────────────
    for i, row in enumerate(new_rows):
        row['ai_summary'] = summarise(row['description'])
        print(f"  [{i+1}/{len(new_rows)}] [{row['section']}] {row['title'][:55]}")
        print(f"        {row['ai_summary'][:80]}...")

    # ── Append to CSV ─────────────────────────────────────
    append_to_csv(new_rows)
    print(f'Total in dataset: {len(load_seen_urls())}')
    print(f'Next cycle in {FETCH_INTERVAL_MINUTES} min.')

In [ ]:
# ── Single test run ──────────────────────────────────────
run_cycle()

In [ ]:
# ── Continuous pipeline (uncomment to start) ─────────────
# def start_pipeline():
#     run_cycle()
#     schedule.every(FETCH_INTERVAL_MINUTES).minutes.do(run_cycle)
#     while True:
#         schedule.run_pending()
#         time.sleep(30)
# start_pipeline()

In [ ]:
# ═══════════════════════════════════════════════════════
# LIVE FEED VIEWER — grouped by section
# ═══════════════════════════════════════════════════════

if os.path.exists(OUTPUT_CSV):
    df = pd.read_csv(OUTPUT_CSV)

    # Only show articles from the last 48 hours
    def recompute_age(row):
        dt = parse_dt(str(row.get('published_at', '')))
        return age_hours(dt)
    df['age_hours'] = df.apply(recompute_age, axis=1)
    df = df[df['age_hours'] <= MAX_AGE_HOURS].sort_values('age_hours')

    print(f'\nLIVE FEED — {len(df)} articles from last {MAX_AGE_HOURS}h')
    print(f'As of: {datetime.now().strftime("%Y-%m-%d %H:%M")} AEST')

    for section in SECTION_PRIORITY:
        subset = df[df['section'] == section] if 'section' in df.columns else pd.DataFrame()
        if subset.empty: continue
        print(f'\n{"─"*60}')
        print(f'[{section.upper()}] — {len(subset)} articles')
        for _, row in subset.head(5).iterrows():
            print(f"  [{row.get('website_name','?')}] {row.get('title','?')}")
            print(f"  Summary : {row.get('ai_summary','(not yet summarised)')}")
            print(f"  {row.get('url','')[:80]}")
            print()
else:
    print(f'{OUTPUT_CSV} not found. Run a cycle first.')